In [1]:
import pprint
import logging

logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

documents = [
    "Human machine interface for lab abc computer applications",
    "A survey of user opinion of computer system response time",
    "The EPS user interface management system",
    "System and human system engineering testing of EPS",
    "Relation of user perceived response time to error measurement",
    "The generation of random binary unordered trees",
    "The intersection graph of paths in trees",
    "Graph minors IV Widths of trees and well quasi ordering",
    "Graph minors A survey",
]

In [2]:
# Create a set of frequent words
stoplist = set('for a of the and to in'.split(' '))
# Lowercase each document, split it by white space and filter out stopwords
texts = [[word for word in document.lower().split() if word not in stoplist]
         for document in documents]
# Count word frequencies
from collections import defaultdict
frequency = defaultdict(int)
for text in texts:
    for token in text:
        frequency[token] += 1
# Only keep words that appear more than once
texts = [[token for token in text if frequency[token] > 1] for text in texts]
pprint.pprint(texts)

[['human', 'interface', 'computer'],
 ['survey', 'user', 'computer', 'system', 'response', 'time'],
 ['eps', 'user', 'interface', 'system'],
 ['system', 'human', 'system', 'eps'],
 ['user', 'response', 'time'],
 ['trees'],
 ['graph', 'trees'],
 ['graph', 'minors', 'trees'],
 ['graph', 'minors', 'survey']]


In [4]:
from gensim import corpora

dictionary = corpora.Dictionary(texts)
dictionary.save("deerwester.dict")
print(dictionary)

2026-08-20 15:21:53,102 : INFO : adding document #0 to Dictionary<0 unique tokens: []>
2026-08-20 15:21:53,104 : INFO : built Dictionary<12 unique tokens: ['computer', 'human', 'interface', 'response', 'survey']...> from 9 documents (total 29 corpus positions)
2026-08-20 15:21:53,105 : INFO : Dictionary lifecycle event {'msg': "built Dictionary<12 unique tokens: ['computer', 'human', 'interface', 'response', 'survey']...> from 9 documents (total 29 corpus positions)", 'datetime': '2026-08-20T15:21:53.105419', 'gensim': '4.4.0', 'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26100-SP0', 'event': 'created'}
2026-08-20 15:21:53,107 : INFO : Dictionary lifecycle event {'fname_or_handle': 'deerwester.dict', 'separately': 'None', 'sep_limit': 10485760, 'ignore': frozenset(), 'datetime': '2026-08-20T15:21:53.107419', 'gensim': '4.4.0', 'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit 

Dictionary<12 unique tokens: ['computer', 'human', 'interface', 'response', 'survey']...>


In [5]:
print(dictionary.token2id)

{'computer': 0, 'human': 1, 'interface': 2, 'response': 3, 'survey': 4, 'system': 5, 'time': 6, 'user': 7, 'eps': 8, 'trees': 9, 'graph': 10, 'minors': 11}


In [6]:
new_doc = "Human computer interaction"
new_vec = dictionary.doc2bow(new_doc.lower().split())
print(new_vec)

[(0, 1), (1, 1)]


In [7]:
corpus = [dictionary.doc2bow(text) for text in texts]
corpora.MmCorpus.serialize("deerwester.mm",corpus)
print(corpus)

2026-08-20 15:27:18,391 : INFO : storing corpus in Matrix Market format to deerwester.mm
2026-08-20 15:27:18,394 : INFO : saving sparse matrix to deerwester.mm
2026-08-20 15:27:18,395 : INFO : PROGRESS: saving document #0
2026-08-20 15:27:18,397 : INFO : saved 9x12 matrix, density=25.926% (28/108)
2026-08-20 15:27:18,583 : INFO : saving MmCorpus index to deerwester.mm.index


[[(0, 1), (1, 1), (2, 1)], [(0, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1)], [(2, 1), (5, 1), (7, 1), (8, 1)], [(1, 1), (5, 2), (8, 1)], [(3, 1), (6, 1), (7, 1)], [(9, 1)], [(9, 1), (10, 1)], [(9, 1), (10, 1), (11, 1)], [(4, 1), (10, 1), (11, 1)]]


In [9]:
file_name = 'mycorpus.txt'
class MyCorpus:
    def __init__(self,file_name):
        self.filename = file_name

    def __iter__(self):
        for line in open(file_name):
            yield dictionary.doc2bow(line.lower().split())
corpus_memory_friendly = MyCorpus(file_name)  # doesn't load the corpus into memory!
print(corpus_memory_friendly)


In [10]:
for vector in corpus_memory_friendly:
    print(vector)

[(0, 1), (1, 1), (2, 1)]
[(0, 1), (3, 1), (4, 1), (5, 1), (6, 1), (7, 1)]
[(2, 1), (5, 1), (7, 1), (8, 1)]
[(1, 1), (5, 2), (8, 1)]
[(3, 1), (6, 1), (7, 1)]
[(9, 1)]
[(9, 1), (10, 1)]
[(9, 1), (10, 1), (11, 1)]
[(4, 1), (10, 1), (11, 1)]


In [11]:
# 从文件中加载文档和停用词
dictionary = corpora.Dictionary(line.lower().split() for line in open("mycorpus.txt"))
stoplist = [word for stopwords in open('stoplist.txt') for word in stopwords.split()]
# remove stop words and words that appear only once
stopids=[
    dictionary.token2id[stopword]
    for stopword in stoplist
    if stopword in dictionary.token2id
]

once_ids = [tokenid for tokenid,docfreq in dictionary.dfs.items() if docfreq ==1 ]
# 过滤
dictionary.filter_tokens(stopids + once_ids)
# remove gaps in id sequence after words that were removed
dictionary.compactify()
print(dictionary)

2026-08-20 16:07:38,376 : INFO : adding document #0 to Dictionary<0 unique tokens: []>
2026-08-20 16:07:38,378 : INFO : built Dictionary<42 unique tokens: ['abc', 'applications', 'computer', 'for', 'human']...> from 9 documents (total 69 corpus positions)
2026-08-20 16:07:38,379 : INFO : Dictionary lifecycle event {'msg': "built Dictionary<42 unique tokens: ['abc', 'applications', 'computer', 'for', 'human']...> from 9 documents (total 69 corpus positions)", 'datetime': '2026-08-20T16:07:38.379102', 'gensim': '4.4.0', 'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26100-SP0', 'event': 'created'}


Dictionary<12 unique tokens: ['computer', 'human', 'interface', 'response', 'survey']...>


In [12]:
from gensim.models import Word2Vec
# define training data
sentences = [['this', 'is', 'the', 'first', 'sentence', 'for', 'word2vec'],
            ['this', 'is', 'the', 'second', 'sentence'],
            ['yet', 'another', 'sentence'],
            ['one', 'more', 'sentence'],
            ['and', 'the', 'final', 'sentence']]
# train model
model = Word2Vec(sentences, min_count=1)
# summarize the loaded model
print(model)
# summarize vocabulary
words = list(model.wv.key_to_index)
print(words)
# access vector for one word
print(model.wv['sentence'])   
# save model
model.save('model.bin')
# load model
new_model = Word2Vec.load('model.bin')
print(new_model)

2026-08-20 16:22:34,542 : INFO : collecting all words and their counts
2026-08-20 16:22:34,543 : INFO : PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2026-08-20 16:22:34,545 : INFO : collected 14 word types from a corpus of 22 raw words and 5 sentences
2026-08-20 16:22:34,546 : INFO : Creating a fresh vocabulary
2026-08-20 16:22:34,547 : INFO : Word2Vec lifecycle event {'msg': 'effective_min_count=1 retains 14 unique words (100.00% of original 14, drops 0)', 'datetime': '2026-08-20T16:22:34.547237', 'gensim': '4.4.0', 'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26100-SP0', 'event': 'prepare_vocab'}
2026-08-20 16:22:34,549 : INFO : Word2Vec lifecycle event {'msg': 'effective_min_count=1 leaves 22 word corpus (100.00% of original 22, drops 0)', 'datetime': '2026-08-20T16:22:34.549238', 'gensim': '4.4.0', 'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD

Word2Vec<vocab=14, vector_size=100, alpha=0.025>
['sentence', 'the', 'is', 'this', 'final', 'and', 'more', 'one', 'another', 'yet', 'second', 'word2vec', 'for', 'first']
[-5.3622725e-04  2.3643136e-04  5.1033497e-03  9.0092728e-03
 -9.3029495e-03 -7.1168090e-03  6.4588725e-03  8.9729885e-03
 -5.0154282e-03 -3.7633716e-03  7.3805046e-03 -1.5334714e-03
 -4.5366134e-03  6.5540518e-03 -4.8601604e-03 -1.8160177e-03
  2.8765798e-03  9.9187379e-04 -8.2852151e-03 -9.4488179e-03
  7.3117660e-03  5.0702621e-03  6.7576934e-03  7.6286553e-04
  6.3508903e-03 -3.4053659e-03 -9.4640139e-04  5.7685734e-03
 -7.5216377e-03 -3.9361035e-03 -7.5115822e-03 -9.3004224e-04
  9.5381187e-03 -7.3191668e-03 -2.3337686e-03 -1.9377411e-03
  8.0774371e-03 -5.9308959e-03  4.5162440e-05 -4.7537340e-03
 -9.6035507e-03  5.0072931e-03 -8.7595852e-03 -4.3918253e-03
 -3.5099984e-05 -2.9618145e-04 -7.6612402e-03  9.6147433e-03
  4.9820580e-03  9.2331432e-03 -8.1579173e-03  4.4957981e-03
 -4.1370760e-03  8.2453608e-04  8.498